In [ ]:
#| hide
from vishalakshi import *
from fastcore.all import *
try:
    from litert_lm import set_min_log_severity, Backend
    from rishi.litert import gemma4_e2b, gemma4_e4b
    set_min_log_severity(5)
except ImportError: pass

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# vishalakshi

> one vault for everything you read: web, papers, video, files and code, searchable together and answerable by a local or hosted model

One SQLite file. A question crosses papers, pages, talks, notes, and the source tree on disk.

[fossick](https://github.com/vedicreader/fossick) fetches, [litesearch](https://github.com/Karthik777/litesearch) stores and retrieves, [rishi](https://github.com/vedicreader/rishi) answers, [kosha](https://github.com/vedicreader/kosha) indexes code, [rgapi](https://github.com/AnswerDotAI/rgapi) greps what nothing has indexed yet. A `Vault` is a litesearch [`Index`](https://Karthik777.github.io/litesearch/api.html#index); retrieval defaults come from there.


## Install

```sh
pip install vishalakshi
```


## The loop

`Vault()` with no argument uses `~/.vishalakshi/vault.db`. This page uses a throwaway file against this repository's own docs and source.


In [ ]:
from tempfile import mkdtemp
from litesearch import repo_root
from vishalakshi import Vault

root = str(repo_root() or Path('..'))   # works from nbs/ or the repo root
v = Vault(Path(mkdtemp())/'vault.db')
v.enc.note


'minishlab/potion-multilingual-128M (256d, float16, model2vec)'

`add` takes a directory, a file, or text. `grab` routes an arXiv id, YouTube link, GitHub repo, PDF, file, or directory.

In [ ]:
v.add(root)                       # README.md and every notebook under nbs/
v.note('federate fuses the legs by rank because they share no vector space: the vault embeds '
       'prose, kosha embeds identifiers, ripgrep embeds nothing.', tags=['retrieval', 'design'])
v.stats()

{'docs': 19,
 'nodes': 166,
 'chunks': 672,
 'encoder': 'model2vec',
 'entities': 0,
 'path': '/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp0ufdxnt4/vault.db',
 'by_kind': {'notebook': 12, 'md': 5, 'txt': 1, 'note': 1}}

Default model: `gemma-4-E2B` on LiteRT GPU, no API key. Name any rishi model id/path; `chat_kw=` reaches its constructor. `$VISHALAKSHI_MODEL` replaces the id; `$VISHALAKSHI_GPU=0` puts LiteRT on CPU. Context budget is `sections=4`, `max_chars=1500`.


In [ ]:
#| eval: false
r = v.ask('why are rankings fused instead of distances?')
print(r.model, '·', r.runtime)
print(r.answer)
print(L(r.cited).map(lambda c: f"{c['n']} {c['breadcrumb']}"))
if (hit := first(r.cited)): print(v.read(hit['node_id'])['text'][:400])

litert/litert-community/gemma-4-E2B-it-litert-lm · litert
The provided sections do not explicitly state why rankings are fused instead of distances. However, one section mentions that `federate` fuses rankings with RRF and never distances [2].
['2 03 code › Fusing legs that share no vector space']
The vault embeds prose, kosha embeds identifiers, ripgrep embeds nothing. Legs share no vector space, so `federate` fuses rankings with RRF and never distances.


#| export
def _prose(v, q, n, kind=None, source:str='prose') -> L:
    'Vault sections, normalised to the federated row shape. `source` names the shelf they came from.'
    return L(v.sections(q, limit=n, kind=kind)).map(
        lambda 


## PII and noise

`ask`, `extract` and `explain` share one gate: 0.962 precision at recall 1.000 over emails, cards, keys and street lines (`evals/pii.py`). Names are opt-in (`ner=True`, honorific-anchored); `scanned_ner` says whether anything looked. Answers are re-scanned with names on.

Junk is separate: `suggest_noisy` scores 0.988 AUC with no labels; `mark_noisy` excludes. See [pii](09_pii.ipynb) and [quality](10_quality.ipynb).


In [ ]:
from vishalakshi.pii import pii_report

r = pii_report('Invoice for ada@example.com. Card 4111 1111 1111 1111. KEY=sk-abcdefghijklmnopqrstuvwxyz123456')
r.has_pii, r.identifying, r.kinds

(True,
 {'email': 1, 'card': 1, 'secret': 1},
 {'email': 1, 'card': 1, 'secret': 1})

In [ ]:
# a person's judgement, over the top of the arithmetic
v.add('A letter about Jane, and what she said on Tuesday.',
      title='letter', source='/inbox/letter.md')
v.add('Cookie policy. All rights reserved. Privacy. Terms. Contact us.',
      title='footer', source='/inbox/footer.md')
v.mark_pii('/inbox/letter.md', reason='names a person')     # private even though nothing matched
v.mark_noisy('/inbox/footer.md', reason='site furniture')   # out of search, sections, context, ask

r = v.pii('/inbox/letter.md', ner=True)   # scanned_ner says whether names were looked for
r.scanned_ner, r.detected, r.has_pii, any('footer' in h['breadcrumb'] for h in v.search('cookie policy'))

| `pii=` | what happens |
|---|---|
| `local` (default) | local model answers; shape and quantity, not detail. Structured `fields` are scrubbed too |
| `redact` | mask recognised spans, then any model may answer. Names are not masked |
| `refuse` | return the finding, no answer |
| `off` | do not look |

```python
v.mark_not_pii(doc_id, reason='my own invoice')
v.accept_noisy(k=10, reason='reviewed')   # mark the current top suggestions
v.fit_noise(save=True); v.use_noise(True) # optional; fitted blend 0.996 AUC
```


## Paperwork

`kind` is how a document arrived. `categorize` says what it is (cues first; model only on ties). `extract` returns fields; omit `schema` and the doctype picks one. `ask_doc` / `ref=` answer from named documents with the vault behind them. Depth on [extract](06_extract.ipynb) and [ask](02_ask.ipynb).


In [ ]:
INVOICE = '''# INVOICE
Invoice No: ACM-2024-0117
Date: 2024-03-01
Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd
Total due: 1,240.00 EUR
Due date: 2024-03-31
Items: widgets x10 @ 124.00
'''
v.add(INVOICE, title='Acme invoice 0117', source='/inbox/acme-0117.md')
v.categorize_all(llm='never')
v.doctypes()

{'code': 13, 'documentation': 4, 'invoice': 1, 'paper': 1, 'other': 1}

In [ ]:
#| eval: false
from dataclasses import fields
from vishalakshi.extract import SCHEMAS, as_schema

print(L(SCHEMAS), L(fields(as_schema('invoice'))).attrgot('name'))
e = v.extract('/inbox/acme-0117.md')                    # doctype picks schema
a = v.ask_doc('/inbox/acme-0117.md', 'what is owed, to whom, and by when?',
              schema='amount:float, payee:str, due:str')
e.schema, e.fields, a

[{'invoice': <class 'vishalakshi.extract.Invoice'>, 'purchase_order': <class 'vishalakshi.extract.Invoice'>, 'quote': <class 'vishalakshi.extract.Invoice'>, 'receipt': <class 'vishalakshi.extract.Receipt'>, 'catalogue': <class 'vishalakshi.extract.Catalogue'>, 'contract': <class 'vishalakshi.extract.Contract'>, 'resume': <class 'vishalakshi.extract.Resume'>, 'paper': <class 'vishalakshi.extract.Paper'>, 'meeting_notes': <class 'vishalakshi.extract.MeetingNotes'>, 'other': <class 'vishalakshi.extract.Summary'>}] ['number', 'date', 'due_date', 'vendor', 'vendor_tax_id', 'bill_to', 'ship_to', 'currency', 'subtotal', 'tax', 'total', 'payment_terms', 'items']


/Users/71293/code/personal/orgs/vishalakshi/vishalakshi/extract.py:580: UserWarning: ValueError on a constrained call for Answer (model neither called the tool nor returned JSON; reply: 'The invoice details are as follows:\n*   **Invoice No:** ACM-2024-0117 [1]\n*   **Date:** 202); retrying as a JSON reply.
  warnings.warn(f'{type(e).__name__} on a constrained call for {schema.__name__} '


('Invoice',
 {'number': 'ACM-2024-0117',
  'date': '2024-03-01',
  'due_date': '',
  'vendor': 'Acme Supplies Ltd',
  'vendor_tax_id': '',
  'bill_to': 'Contoso GmbH, Berlin',
  'ship_to': '',
  'currency': 'EUR',
  'subtotal': 1240.0,
  'tax': 0,
  'total': 1240.0,
  'payment_terms': '',
  'items': ['widgets x10 @ 124.00']},
 {'question': 'what is owed, to whom, and by when?',
  'model': 'litert/litert-community/gemma-4-E2B-it-litert-lm',
  'runtime': 'litert',
  'context': {'results': [{'node_id': '4c1bdc4b329e3c3d#0', 'title': 'Acme invoice 0117', 'doc_id': '4c1bdc4b329e3c3d', 'breadcrumb': 'Acme invoice 0117', 'filename': '/inbox/acme-0117.md', 'pages': None, 'text': '# INVOICE\nInvoice No: ACM-2024-0117\nDate: 2024-03-01\nBill to: Contoso GmbH, Berlin\nFrom: Acme Supplies Ltd\nTotal due: 1,240.00 EUR\nDue date: 2024-03-31\nItems: widgets x10 @ 124.00'}, {'node_id': 'e8024c5880508178#0', 'title': '05 mcp', 'doc_id': 'e8024c5880508178', 'breadcrumb': '05 mcp', 'filename': None, 'pag

## Code

`index_code` fills kosha; `context(..., code=n)` then appends code sections beside prose. `grep` is ripgrep on disk (no index). `federate` is on [code](03_code.ipynb).


In [ ]:
#| eval: false
v.index_code(root)                # fills .kosha/; context then appends code sections
c = v.context('where does the entity graph get rebuilt?', sections=3, related=0, code=3, dir=root)
# code hits have no node_id; their handle is path:line on disk
c.code, L(c.results).filter(lambda r: r.node_id is None).attrgot('breadcrumb')

parse files from /Users/71293/code/personal/orgs/vishalakshi: 100%|██████████| 18/18 [00:00<00:00, 225.97it/s]


(3,
 ['repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/core.py:520', 'grep › README.md:119', 'repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/core.py:547'])

In [ ]:
L(v.grep('rrf_all', root, limit=4)).attrgot('where')   # ripgrep; no kosha needed

['README.md:125', 'nbs/index.ipynb:573', 'vishalakshi/code.py:12', 'vishalakshi/code.py:128']

## Watches

An action is an acquisition method name. `poll()` is the tick (cron, scheduler, or button). See [acquire](01_acquire.ipynb) for `harvest` and `apis`.


In [ ]:
v.watch('https://example.com/changelog', action='url', every='6h')
v.watch('late chunking retrieval', action='web', every='1d', n=5)
v.watch('Re-read the evals', action='remind', every='1w')
L(v.watches()).map(lambda w: (w['action'], w['target'][:34], w['every'], w['params']))

[('url', 'https://example.com/changelog', 21600.0, {}), ('web', 'late chunking retrieval', 86400.0, {'n': 5}), ('remind', 'Re-read the evals', 604800.0, {})]

## The rest

| page | what is on it |
|---|---|
| [core](00_core.ipynb) | `Vault`, shelves, `context`, entity graph, `document` |
| [acquire](01_acquire.ipynb) | `grab`, `url`, `web`, `crawl`, `arxiv`, `pdf`, `youtube`, `github`, `apis`, `harvest`, watches |
| [ask](02_ask.ipynb) | `ask`, `ask_doc`, citations, model plumbing, CachedChat |
| [code](03_code.ipynb) | kosha, `symbol`, `where_to_add`, `grep`, `federate` |
| [cli](04_cli.ipynb) | every `Vault` method as a command |
| [mcp](05_mcp.ipynb) | `vishalakshi-mcp` |
| [extract](06_extract.ipynb) | `categorize`, `extract`, `extract_all`, schemas |
| [concepts](07_concepts.ipynb) | encoders, shelves, backends, `reshelf` |
| [skill](08_skill.ipynb) | agent cheat sheet (exported skill) |
| [pii](09_pii.ipynb) | patterns, checksums, `secret`, `mark_pii` / `mark_not_pii`, `redact` |
| [quality](10_quality.ipynb) | `suggest_noisy` / `accept_noisy`, `fit_noise`, ranker |

```sh
vishalakshi grab https://example.com/post
vishalakshi ask "why does late chunking help"
```

| variable | what it sets |
|---|---|
| `$VISHALAKSHI_VAULT` | the vault file |
| `$VISHALAKSHI_MODEL` | the model `ask` uses |
| `$VISHALAKSHI_PII_MODEL` | local model when sections are private |
| `$VISHALAKSHI_GPU=0` | put LiteRT on the CPU |
| `$VISHALAKSHI_OFFLINE` | never download; hashing encoder |

MCP client config is on [mcp](05_mcp.ipynb). Retrieval trade-offs and measured defaults are in [concepts](07_concepts.ipynb) and [`evals/RESULTS.md`](../evals/RESULTS.md).


## Development

The notebooks in `nbs/` are the source; the modules are generated.


```sh
pip install -e .
nbdev-prepare
```
